<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/Gradient_Descent_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
import pandas as pd
pd.set_option('display.max_columns',100)
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (22, 5)
plt.rcParams['figure.dpi'] = 100
import sqlite3
from sklearn.metrics import r2_score, roc_curve, auc
from urllib.request import urlretrieve
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [ ]:
medical_charges_url = 'https://raw.githubusercontent.com/JovianML/opendatasets/master/data/medical-charges.csv'

medical_df = pd.read_csv(medical_charges_url)
medical_df.head()

In [ ]:
medical_df.shape

In [ ]:
medical_df.describe()


In [ ]:
medical_df.info()


In [ ]:
conn = sqlite3.connect('database.db' )
# Write the DataFrame to a table in the database
medical_df.to_sql('data', conn, if_exists='replace', index=False)


In [ ]:
sns.boxplot(data=medical_df, x='sex', y='charges')
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.title('Distribution of Charges by Gender')
plt.xlabel('Gender')
plt.ylabel('Charges')
plt.show()

In [ ]:
sns.catplot(data=medical_df,x= 'sex',y='charges', kind='swarm', hue='smoker', aspect =2, height=8)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.annotate(f' Average Medical Charges ($)', (.4, medical_df['charges'].mean()+900), fontsize=14,color='red')
plt.title('Insurance Charges Gender Wise', fontsize=20)
plt.show()

In [ ]:
sns.catplot(data=medical_df,x= 'region',y='charges', kind='box', aspect =2, height=8)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=4, zorder=1, color='red')
plt.title('Medical Charges Region Wise', fontsize=18)
plt.xlabel('Region')
plt.ylabel('Medical Charges ($)')
plt.show()


In [ ]:
sns.barplot(data=medical_df,x= 'smoker',y='charges', hue='smoker', alpha=0.7, dodge=False)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='red')
plt.title('Insurance Charges smoker Wise', fontsize=18)
plt.ylabel('Medical Charges ($)')
plt.show()

In [ ]:
query = '''

select sex, smoker, round(sum(charges),2) as MedicalCharges
from data
group by sex, smoker

'''

temp = pd.read_sql(query, conn)
female_df = temp[temp['sex']=='female']
male_df = temp[temp['sex']=='male']

plt.subplot(1, 2, 1)
plt.bar(x=female_df['smoker'], height=female_df['MedicalCharges'], color='#97E7E1')
plt.xlabel('Is Smoker?')
plt.ylabel('MedicalCharges')
plt.title('Distribution of Medical Charges, Females')

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.subplot(1, 2, 2)
plt.bar(x=male_df['smoker'], height=male_df['MedicalCharges'], color='#6AD4DD')
plt.xlabel('Is Smoker?')
plt.ylabel('MedicalCharges')
plt.title('Distribution of Medical Charges, Males')

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
query = '''
SELECT
    CASE
        WHEN age < 30 THEN 'Young'
        WHEN age < 60 THEN 'Adult'
        ELSE 'Elderly'
    END AS age_group,
    ROUND(AVG(charges), 2) AS AvgMedicalCharges
FROM
    data
where smoker='yes'
GROUP BY
    CASE
        WHEN age < 30 THEN 'Young'
        WHEN age < 60 THEN 'Adult'
        ELSE 'Elderly'
    END;

'''

temp = pd.read_sql(query, conn)
custom_palette = ['#C9CCD5', '#E23E57', '#C9CCD5']

sns.barplot(data=temp,x= 'age_group',y='AvgMedicalCharges', hue='age_group', dodge=False, palette=custom_palette)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.title('Smokers AvgMedicalCharges Age Bracket Wise', fontsize=18)
plt.ylabel('Avg Medical Charges ($)')
plt.show()


In [ ]:
query = '''
SELECT
    CASE
        WHEN age < 30 THEN 'Young'
        WHEN age < 60 THEN 'Adult'
        ELSE 'Elderly'
    END AS age_group,
    ROUND(AVG(charges), 2) AS AvgMedicalCharges
FROM
    data
where smoker='no'
GROUP BY
    CASE
        WHEN age < 30 THEN 'Young'
        WHEN age < 60 THEN 'Adult'
        ELSE 'Elderly'
    END;

'''

temp = pd.read_sql(query, conn)

sns.barplot(data=temp,x= 'age_group',y='AvgMedicalCharges', hue='age_group', dodge=False,palette=custom_palette )
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.title('Non Smokers AvgMedicalCharges Age Bracket Wise', fontsize=18)
plt.ylabel('Avg Medical Charges ($)')
plt.show()


In [ ]:
fig, ax = plt.subplots()
N, bins, patches = ax.hist(np.array(medical_df.charges), edgecolor='white', color='lightgray',linewidth=5, alpha=0.7)
for i in range(0,1):
    patches[i].set_facecolor('orange')
    plt.title('Medical Charges Histogram', fontsize=18)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.xlabel('Medical Charges ($)')
    plt.ylabel('Count')
    plt.axvline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='blue')
    plt.annotate(f' Average Medical Charges ($)', (13500, 500), fontsize=14,color='red')
    plt.show()

In [ ]:
sns.scatterplot(y=medical_df['charges'], x=medical_df['age'], hue=medical_df['smoker'], alpha=0.5)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.annotate(f'Average Medical Charges ($)', (45, 13900), fontsize=14,color='red')
plt.title('Age Wise Medical Charges Distribution', fontsize=18)
plt.ylabel('Medical Charges ($)')
plt.show()


In [ ]:
sns.scatterplot(y=medical_df['bmi'], x=medical_df['charges'], hue=medical_df['smoker'], alpha=0.5)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
xticks = plt.gca().get_xticks()
xlabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in xticks]
plt.gca().set_xticklabels(xlabels)

plt.axvline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.annotate(f'Average Medical Charges ($)', (13400, 52), fontsize=14, color='Red')

plt.title('BMI & Medical Charges relation', fontsize=18)
plt.xlabel('Medical Charges ($)')
plt.ylabel('BMI')
plt.show()

In [ ]:
sns.relplot(data=medical_df, x='children', y='charges',  hue='smoker', aspect = 2, height=8)
yticks = plt.gca().get_yticks()
ylabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in yticks]
plt.gca().set_yticklabels(ylabels)
plt.axhline(medical_df['charges'].mean(), linestyle='--', lw=2, zorder=1, color='black')
plt.annotate(f'Average Medical Charges ($)', (4.05 , 13900), fontsize=12, color='red')
plt.title('Children & Medical Charges relation', fontsize=18)
plt.ylabel('Medical Charges ($)')
plt.xlabel('Number of Children')
plt.show()

In [ ]:
sns.boxplot(x=medical_df['charges'])
xticks = plt.gca().get_xticks()
xlabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in xticks]
plt.gca().set_xticklabels(xlabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(medical_df['charges'].mean(), linestyle='--', lw=4, zorder=1, color='red')
plt.annotate(f'Average Medical Charges ($)', (13800, 0.47), fontsize=15, color='red')
plt.title('Outliers in the data', fontsize=18)
plt.xlabel('Medical Charges ($)')
plt.show()

In [ ]:
ax = sns.histplot(medical_df['charges'], kde=True, color='lightgray')
xticks = plt.gca().get_xticks()
xlabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in xticks]
plt.gca().set_xticklabels(xlabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False);
ax.lines[0].set_color('red')
plt.axvline(medical_df['charges'].mean(), linestyle='--', lw=3, zorder=1, color='blue')
plt.annotate(f'Average Medical Charges ($)', (13700, 200), fontsize=15, color='blue')
plt.title('Detecting Outliers Using Histogram', fontsize=18)
plt.xlabel('Medical Charges ($)')
plt.show()

In [ ]:
data = sorted(medical_df['charges'].values)

data_mean, data_std = np.mean(data), np.std(data)
cut_off = data_std * 3

lower, upper = data_mean - cut_off, data_mean + cut_off

print('Cut Off =', round(cut_off, 3))
print('Lower =', round(lower, 3))
print('Upper =', round(upper, 3))

In [ ]:
ax = sns.histplot(medical_df['charges'], kde=True, color='lightgray')
xticks = plt.gca().get_xticks()
xlabels = [f"{round(i / 1000)}k" if i != 0 else "0" for i in xticks]
plt.gca().set_xticklabels(xlabels)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False);
ax.lines[0].set_color('red')
plt.axvline(data_mean, linestyle='--', lw=2, zorder=1, color='orange')
plt.annotate(f'Average', (data_mean+500, 175), fontsize=15, color='blue')

plt.axvline(upper, linestyle='--', lw=2, zorder=1, color='orange')
plt.annotate(f'Upper', (upper+500, 175), fontsize=15, color='blue')

plt.axvline(cut_off, linestyle='--', lw=2, zorder=1, color='orange')
plt.annotate(f'Cut Off', (cut_off+500, 175), fontsize=15, color='blue')

plt.title('Detecting Outliers', fontsize=18)
plt.xlabel('Medical Charges ($)')
plt.show()


In [ ]:
medical_df = medical_df[medical_df['charges'] < upper]
medical_df = medical_df[medical_df['charges'] > lower]
print('The shape of our dataframe after the Outlier Removal is', medical_df.shape)

In [ ]:
df = medical_df.copy()


In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_cols = df.select_dtypes(include='object').columns
encoder.fit(df[cat_cols])

In [ ]:
onehot =encoder.transform(df[cat_cols])
encoder.categories_

In [ ]:
encoded_cols = [['female', 'male','smokerno', 'smokeryes', 'northeast', 'northwest', 'southeast', 'southwest']]
df[['female', 'male','smokerno', 'smokeryes', 'northeast', 'northwest', 'southeast', 'southwest']] = onehot
df.drop(cat_cols, axis=1, inplace=True )

In [ ]:
df.head()

In [ ]:
scaler = MinMaxScaler()


In [ ]:
y = df['charges'].values

X = df.drop('charges', axis=1).values

In [ ]:
print('The shape of independent variables data is',X.shape)
print('The shape of the target variable data is',y.shape)

In [ ]:
def EarlyStopping(loss):
    for i in range(1, len(loss)):
        yield (loss[i-1], loss[i])

def batch_size(batchsize, X):
    batches = round(X.shape[0] // batchsize)
    return batches

def regression_gradient_descent(X_train, y_train, m, b):
    # Predictions using the linear equation y = mx + b
    yhat = np.dot(X_train, m) + b

    MSE = (np.sum((y_train - yhat)**2)) / N
    r_squared = r2_score(y_train, yhat)

    loss_slope_b = -(2/N) * sum(y_train - yhat)

    loss_slope_m = -(2/N) * (np.dot((y_train - yhat), X_train))

    m = m - (learning_rate * loss_slope_m)
    b = b - (learning_rate * loss_slope_b)

    return m, b, MSE, r_squared

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
np.random.seed(0)

N = X.shape[0]

learning_rate = 0.2
decay_rate = 0.01
LR = []
ValidationLoss = []
Trainingloss = []  #
batchsize = 30

# Initialize the intercept and slope
Intercept = []
Slope = []
m = np.ones(X.shape[1])
b = 1
print('The initial Value of w and b are', m, b)

# Calculate the number of batches
batches = batch_size(batchsize, X)

num_epochs =  2000

# Loop over multiple epochs for training
for i in range(num_epochs):
    epoch = i

    # Loop over batches within the current epoch
    for j in range(batches):
        if i == 0:
            # Initial epoch updates (no decay rate applied)
            if j % batchsize == 0:
                learning_rate = learning_rate  # Maintain initial learning rate
                np.random.seed(0)
                np.random.shuffle([X_train_scaled, y_train])
                m, b, MSE, r_squared = regression_gradient_descent(X_train_scaled, y_train, m, b)
                m_test, b_test, MSE_test, r_squared_test = regression_gradient_descent(X_test_scaled, y_test, m, b)
            else:
                m = m
                b = b
        else:
            # Updates with decay rate applied
            if j % batchsize == 0:
                learning_rate = [(1 / (1 + decay_rate)) * learning_rate for j in range(batches)][0]
                np.random.seed(0)
                np.random.shuffle([X_train_scaled, y_train])
                m, b, MSE, r_squared = regression_gradient_descent(X_train_scaled, y_train, m, b)
                m_test, b_test, MSE_test, r_squared_test = regression_gradient_descent(X_test_scaled, y_test, m, b)
            else:
                m = m
                b = b

    # Store values for analysis and tracking
    Intercept.append(b)
    Slope.append(m)
    Trainingloss.append(MSE)
    ValidationLoss.append(MSE_test)
    LR.append(learning_rate)

    if i % 100 == 0:
        print(f'Epoch: {i}/{num_epochs} [==============================] - Loss: {MSE:.2e} - val Loss: {MSE_test:.2e} - r-squared: {round(r_squared, 4)} - val_r-squared: {round(r_squared_test, 4)}')



    for prev, curr in EarlyStopping(ValidationLoss):
        if prev - curr < 1e-6:
            print(f'\n -- Early Stopping at Epoch : {i} val_loss : {np.around(MSE_test, 5)},  val r-squared : {np.around(r_squared_test, 5)} --')
            break
    else:
        continue
    break  # Executed if the inner loop DID break

# Print the final values of slope 'm' and intercept 'b'
print('\nThe final values of w and b are', m, b)

In [ ]:
coefficient = Slope[epoch]
intercept = Intercept[epoch]

y_pred = np.dot(X_test_scaled, coefficient) + intercept

df = pd.DataFrame(y_pred, y_test, columns=['y']).reset_index().rename(columns={'index': 'y', 'y': 'y_pred'})

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 5))
fig.subplots_adjust(hspace=.2, wspace=.3)

ax1.plot(Trainingloss, linestyle='--')
ax1.set_title("Training Loss Plot, Regression")
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')

ax2.plot(ValidationLoss, 'tab:orange', linestyle='dashed', markersize=5)
ax2.set_title("Validation Loss Plot, Regression")
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Loss')

ax3 = sns.regplot(data=df, x=df['y'], y=df['y_pred'], color='lightgray', fit_reg=True)
ax3.lines[0].set_color('red')
ax3.set_title('Targets vs Prediction, Regression')
ax3.set_xlabel('Targets')
ax3.set_ylabel('Prediction')

# Display the plots
plt.show()

In [ ]:
data = load_breast_cancer()
cancerdf = pd.DataFrame(data=data.data, columns=data.feature_names)
cancerdf['target'] = data.target
cols = list(cancerdf.columns)
cancerdf.columns = [i.replace(" ","") for i in cols]
cancerdf.head()

In [ ]:
cancerdf.describe().T


In [ ]:
cancerdf.info()

In [ ]:
cancerdf.to_sql('cancer_data', conn, if_exists='replace', index=False)